# Day 4 · 01 · Indexing and Selection

Almost every pandas task starts with "give me these rows and these columns". This notebook covers the selection tools and the rules that decide which one to use.

**In this notebook**
- `[]`, `.loc` and `.iloc`
- Boolean masks, `isin`, `between`, string tests
- `query`
- A `DatetimeIndex` and partial-string selection
- `MultiIndex`: `.loc`, `xs` and `IndexSlice`
- Assigning values safely under Copy-on-Write
- `where`, `mask`, `reindex` and index set operations

**Dataset:** `orders.csv`, `order_items.csv`, `products.csv`, `customers.csv`

In [1]:
from pathlib import Path

import pandas as pd

DATA = Path("../datasets").resolve()

orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date"])
items = pd.read_csv(DATA / "order_items.csv")
products = pd.read_csv(DATA / "products.csv")
customers = pd.read_csv(DATA / "customers.csv", parse_dates=["signup_date"])

print("pandas", pd.__version__)
for name, frame in [("orders", orders), ("items", items), ("products", products), ("customers", customers)]:
    print(f"{name:<10} {frame.shape}")

pandas 3.1.1
orders     (2000, 6)
items      (4439, 6)
products   (40, 5)
customers  (250, 7)


## 1. Square brackets select columns
- `df["col"]` returns a **Series**.
- `df[["a", "b"]]` (a list inside the brackets) returns a **DataFrame**.
- `df[mask]` with a boolean Series filters **rows**.

For anything that involves rows *and* columns, use `.loc` or `.iloc`.

In [2]:
print(type(products["unit_price"]).__name__, "|", type(products[["unit_price"]]).__name__)
products[["product_name", "category", "unit_price"]].head(3)

Series | DataFrame


,product_name,category,unit_price
0,Wireless Mouse,Electronics,24.99
1,Mechanical Keyboard,Electronics,89.00
2,USB-C Hub,Electronics,39.50


## 2. `.loc` uses labels, `.iloc` uses positions
| | Selects by | Slice end |
| --- | --- | --- |
| `.loc[rows, cols]` | index **labels**, column names, boolean masks | **included** |
| `.iloc[rows, cols]` | integer **positions** from 0 | excluded, like Python lists |

Setting `product_id` as the index gives the rows meaningful labels.

In [3]:
catalog = products.set_index("product_id")

print("one value:", catalog.loc["P004", "product_name"])
catalog.loc["P004":"P007", ["product_name", "unit_price"]]  # P007 is included

one value: Noise-Cancelling Headphones


,product_name,unit_price
product_id,,
P004,Noise-Cancelling Headphones,199.0
P005,Webcam 1080p,59.0
P006,Portable SSD 1TB,109.0
P007,Bluetooth Speaker,49.9


In [4]:
catalog.iloc[3:7, [0, 2]]  # positions 3, 4, 5, 6: the same four rows

,product_name,unit_price
product_id,,
P004,Noise-Cancelling Headphones,199.0
P005,Webcam 1080p,59.0
P006,Portable SSD 1TB,109.0
P007,Bluetooth Speaker,49.9


### The classic trap: integer labels are not positions
After filtering, the index keeps the **original** labels. `.loc[0]` then fails or returns a different row than `.iloc[0]`.

In [5]:
store = orders[orders["channel"] == "store"]

print("index labels of the first rows:", store.index[:5].tolist())
print("first row by position, .iloc[0]:", store.iloc[0]["order_id"])
print("label 0 exists in the index    :", 0 in store.index)

# reset_index(drop=True) renumbers from 0 when the old labels no longer matter.
print("after reset_index              :", store.reset_index(drop=True).index[:5].tolist())

index labels of the first rows: [4, 6, 7, 8, 19]
first row by position, .iloc[0]: 10005
label 0 exists in the index    : False
after reset_index              : [0, 1, 2, 3, 4]


## 3. Boolean masks
A comparison produces a Series of `True`/`False`. Use it to keep the `True` rows.

- Combine with `&` (and), `|` (or), `~` (not).
- Put **parentheses** around each comparison: `&` binds tighter than `==` and `<`.
- Python's `and` / `or` do not work on Series.

In [6]:
mask = (products["category"] == "Electronics") & (products["unit_price"] < 60)
print("rows matching:", mask.sum())
products[mask]

rows matching: 4


,product_id,product_name,category,unit_price,unit_cost
0,P001,Wireless Mouse,Electronics,24.99,13.20
2,P003,USB-C Hub,Electronics,39.50,20.65
4,P005,Webcam 1080p,Electronics,59.00,38.64
6,P007,Bluetooth Speaker,Electronics,49.90,27.41


In [7]:
# isin: membership in a list.  between: a range, both ends included.  ~ negates.
in_range = products["unit_price"].between(20, 30)
books_or_toys = products["category"].isin(["Books", "Toys"])

products.loc[in_range & ~books_or_toys, ["product_name", "category", "unit_price"]]

,product_name,category,unit_price
0,Wireless Mouse,Electronics,24.99
10,Desk Lamp,Home,27.00
13,Vacuum Flask,Home,21.50
15,Storage Box Set,Home,29.00
16,Yoga Mat,Sports,25.00


In [8]:
# String tests live under .str and also return boolean masks.
fischer = customers["name"].str.endswith("Fischer")
indian_city = customers["city"].isin(["Chennai", "Bengaluru"])

customers.loc[fischer & indian_city, ["customer_id", "name", "city", "segment"]]

,customer_id,name,city,segment
20,C0021,Sofia Fischer,Bengaluru,Consumer
70,C0071,Grace Fischer,Chennai,Consumer
212,C0213,Grace Fischer,Chennai,Corporate


## 4. `query`: filters as a readable string
`query` takes the condition as text. Column names are used directly and `@name` refers to a Python variable. It is convenient for long conditions and in method chains.

In [9]:
max_price = 30

products.query("category in ['Books', 'Toys'] and unit_price < @max_price")[["product_name", "category", "unit_price"]]

,product_name,category,unit_price
27,Statistics Primer,Books,27.5
33,Puzzle 1000pc,Toys,16.0
35,Board Game Classic,Toys,28.0
36,Plush Bear,Toys,14.5
39,Kite,Toys,11.0


## 5. Sorting and "top N"
- `sort_values` sorts by one or more columns.
- `nlargest` / `nsmallest` return the top rows directly, without sorting the whole frame.

In [10]:
print(products.nlargest(3, "unit_price")[["product_name", "unit_price"]].to_string(index=False))

products.sort_values(["category", "unit_price"], ascending=[True, False]).groupby("category").head(1)[
    ["category", "product_name", "unit_price"]
]

               product_name  unit_price
Noise-Cancelling Headphones       199.0
                Smart Watch       179.0
              Tennis Racket       129.0


,category,product_name,unit_price
29,Books,Data Systems Design,52.0
3,Electronics,Noise-Cancelling Headphones,199.0
9,Home,Air Fryer,119.0
22,Sports,Tennis Racket,129.0
34,Toys,Remote Control Car,58.0


## 6. A `DatetimeIndex` and partial-string selection
With dates as the (sorted) index, `.loc` accepts **partial date strings**: a year, a month, or a range. Both ends of a range are included.

In [11]:
by_date = orders.set_index("order_date").sort_index()

print("orders in March 2024:", len(by_date.loc["2024-03"]))
print("orders in 2025      :", len(by_date.loc["2025"]))

by_date.loc["2024-12-24":"2024-12-25", ["order_id", "status", "channel"]]

orders in March 2024: 90
orders in 2025      : 729


,order_id,status,channel
order_date,,,
2024-12-24,11248,completed,web
2024-12-24,11249,completed,mobile
2024-12-24,11250,cancelled,web
2024-12-25,11251,completed,store
2024-12-25,11252,completed,store


## 7. `MultiIndex`: several levels of labels
A `groupby` on two columns returns a result indexed by **both**. Each level can be selected on its own.

| Need | Use |
| --- | --- |
| everything under one outer label | `.loc["Books"]` |
| one exact combination | `.loc[("Books", "web")]` |
| one label of an **inner** level | `.xs("web", level="channel")` |
| lists or slices on several levels | `.loc[pd.IndexSlice[..., ...], :]` |
| back to ordinary columns | `.reset_index()` |

In [12]:
sales = items.merge(orders, on="order_id").merge(products[["product_id", "category"]], on="product_id")
sales["revenue"] = sales["quantity"] * sales["unit_price"] * (1 - sales["discount"])

summary = sales.groupby(["category", "channel"])["revenue"].agg(["sum", "count"]).round(2)
summary.head(6)

sum  count
category    channel                 
Books       mobile   24100.05    356
            store    17555.50    252
            web      34975.70    520
Electronics mobile   47613.71    285
            store    36863.99    191
            web      73952.50    455

In [13]:
print(summary.loc["Books"], "\n")
print("Books on the web:", summary.loc[("Books", "web"), "sum"])

summary.xs("web", level="channel")

              sum  count
channel                 
mobile   24100.05    356
store    17555.50    252
web      34975.70    520 

Books on the web: 34975.7


,sum,count
category,,
Books,34975.70,520
Electronics,73952.50,455
Home,27800.78,474
Sports,32493.70,395
Toys,20429.90,306


In [14]:
idx = pd.IndexSlice
summary.loc[idx[["Books", "Toys"], ["store", "web"]], :]

sum  count
category channel                 
Books    store    17555.50    252
         web      34975.70    520
Toys     store     7305.18    110
         web      20429.90    306

## 8. One value: `.at` and `.iat`
For a single cell, `.at[label, column]` and `.iat[row, col]` are the fastest accessors. Use them when reading or writing one value, not inside loops over a whole frame.

In [15]:
print(catalog.at["P004", "unit_price"], catalog.iat[3, 2])

199.0 199.0


## 9. Assigning values
Select the rows and the column **in one `.loc` call**, then assign.

pandas 3 uses **Copy-on-Write**: any subset you take behaves as an independent copy. Two consequences:
- Changing a subset never changes the original frame.
- "Chained" assignment such as `df[mask]["col"] = value` updates a temporary copy and has no effect on `df`.

In [16]:
prices = products.copy()
books = prices["category"] == "Books"

prices.loc[books, "unit_price"] = (prices.loc[books, "unit_price"] * 0.9).round(2)

pd.DataFrame(
    {"before": products.loc[books, "unit_price"], "after": prices.loc[books, "unit_price"]}
).head(4)

,before,after
24,32.0,28.80
25,41.0,36.90
26,48.0,43.20
27,27.5,24.75


In [17]:
# A subset is its own object: editing it leaves `products` untouched.
toys = products[products["category"] == "Toys"]
toys["unit_price"] = 0.0

print("subset  :", toys["unit_price"].unique().tolist())
print("original:", products.loc[products["category"] == "Toys", "unit_price"].head(3).tolist())

subset  : [0.0]
original: [44.0, 16.0, 58.0]


## 10. `where` and `mask`: keep the shape, replace some values
Filtering drops rows. `where` keeps every row and replaces the values that **fail** the condition; `mask` replaces the values that **pass** it.

In [18]:
discount = items["discount"]

pd.DataFrame(
    {
        "discount": discount,
        "where(discount > 0)": discount.where(discount > 0),           # others become NaN
        "mask(discount > 0.1, 0.1)": discount.mask(discount > 0.1, 0.1),  # cap at 10%
    }
).iloc[8:14]

,discount,where(discount > 0),"mask(discount > 0.1, 0.1)"
8,0.15,0.15,0.10
9,0.05,0.05,0.05
10,0.00,NaN,0.00
11,0.00,NaN,0.00
12,0.00,NaN,0.00
13,0.15,0.15,0.10


## 11. `reindex`: conform to a new set of labels
`reindex` returns the data arranged on the labels you give it. Labels that did not exist get `NaN`, or a `fill_value`.

Daily order counts only contain days that had orders. Reindexing on a full calendar exposes the days with none.

In [19]:
daily = orders.groupby("order_date").size()
calendar = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
complete = daily.reindex(calendar, fill_value=0)

print("days with orders :", len(daily))
print("days in calendar :", len(complete))
print("days with zero   :", int((complete == 0).sum()))

days with orders : 529
days in calendar : 547
days with zero   : 18


## 12. An index is a set
Index objects support `difference`, `intersection` and `union`, which answers "which keys are in one table but not the other" in one line.

In [20]:
all_customers = pd.Index(customers["customer_id"])
ordering = pd.Index(orders["customer_id"].unique())

print("customers            :", len(all_customers))
print("with an order        :", len(all_customers.intersection(ordering)))
print("never ordered        :", len(all_customers.difference(ordering)))
print("unknown ids in orders:", len(ordering.difference(all_customers)))

customers            : 250
with an order        : 205
never ordered        : 45
unknown ids in orders: 0


## Summary
| Task | Tool |
| --- | --- |
| Columns | `df["a"]`, `df[["a", "b"]]` |
| Rows and columns by label | `df.loc[rows, cols]` |
| Rows and columns by position | `df.iloc[rows, cols]` |
| Rows by condition | `df[mask]`, `df.loc[mask, cols]`, `df.query("...")` |
| One cell | `df.at[label, col]` |
| Update some rows | `df.loc[mask, col] = value` |
| Replace but keep the shape | `where`, `mask` |
| Align to given labels | `reindex` |

## Key takeaways
- `.loc` is label-based and includes the end of a slice. `.iloc` is position-based and excludes it.
- Wrap each condition in parentheses and combine with `&`, `|`, `~`.
- A sorted `DatetimeIndex` makes date-range selection a one-liner.
- Assign through a single `.loc[rows, col]`. Subsets are independent copies.

## Practice
1. Select completed web orders placed in the second quarter of 2025.
2. From `catalog`, return name and price of every third product using `.iloc`.
3. Using `summary`, show revenue for all categories on the `mobile` channel.
4. Raise the price of all Electronics under 50 by 5% in a copy of `products`.